In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path

from data_loader import build_complete_dataset
from window import create_record_windows

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow.keras.backend as K


In [2]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import pandas as pd

print(np.__version__)
print(pd.__version__)

2.5.3
3.0.6


In [4]:

def extract_window_features(window_df, fs, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time"]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
        signal = data[:, i]
        features[f"{prefix}_{col}_mean"] = np.mean(signal)
        features[f"{prefix}_{col}_std"] = np.std(signal)
        features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal ** 2))
        features[f"{prefix}_{col}_max"] = np.max(signal)
        features[f"{prefix}_{col}_min"] = np.min(signal)
        features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
        features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
        features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
        fft_vals = np.abs(np.fft.rfft(signal))
        freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
        features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals ** 2)
        features[f"{prefix}_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features


In [5]:

def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)

    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))

    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], fs=6700, prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], fs=6700, prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], fs=16000, prefix="mic"))

        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)

    return pd.DataFrame(record_features)


In [6]:

def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)


In [7]:

path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())
print(X_test_df.columns.tolist())

X_train_df_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_test_df_source = X_test_df[X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy()

X_train_df_target = X_train_df[X_train_df["split_label"] == "Normal_Target_Train"].copy()
X_test_df_target = X_test_df[X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy()

metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test_df[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test_df[mic_features].fillna(0)

X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test_df[gyro_features].fillna(0)

X_train_source_acc = X_train_df_source[acc_features].fillna(0)
X_train_source_mic = X_train_df_source[mic_features].fillna(0)
X_train_source_gyro = X_train_df_source[gyro_features].fillna(0)

X_test_source_acc = X_test_df_source[acc_features].fillna(0)
X_test_source_mic = X_test_df_source[mic_features].fillna(0)
X_test_source_gyro = X_test_df_source[gyro_features].fillna(0)

X_train_target_acc = X_train_df_target[acc_features].fillna(0)
X_train_target_mic = X_train_df_target[mic_features].fillna(0)
X_train_target_gyro = X_train_df_target[gyro_features].fillna(0)

X_test_target_acc = X_test_df_target[acc_features].fillna(0)
X_test_target_mic = X_test_df_target[mic_features].fillna(0)
X_test_target_gyro = X_test_df_target[gyro_features].fillna(0)

Train: (102984, 75)
Test : (25984, 75)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [8]:


def build_autoencoder(input_dim, encoding_dim=16):
    input_layer = Input(shape=(input_dim,))

    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(encoding_dim, activation='relu')(encoded)

    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder


In [9]:

def run_autoencoder_pipeline(X_train, X_test, train_df, test_df, sensor_type, encoding_dim=16, epochs=30, batch_size=32,
                             threshold_percentile=99):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = build_autoencoder(input_dim=X_train_scaled.shape[1], encoding_dim=encoding_dim)
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(
        X_train_scaled, X_train_scaled,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    train_pred = model.predict(X_train_scaled, verbose=0)
    train_scores = np.mean(np.power(X_train_scaled - train_pred, 2), axis=1)

    test_pred = model.predict(X_test_scaled, verbose=0)
    test_scores = np.mean(np.power(X_test_scaled - test_pred, 2), axis=1)

    results = pd.DataFrame({
        "segment_id": test_df["segment_id"].values,
        "anomaly_label": test_df["anomaly_label"].values,
        "score": test_scores
    })

    segment_scores = (results.groupby("segment_id").agg(score=("score", "max"),
                                                        anomaly_label=("anomaly_label", "first")).reset_index())
    segment_scores["y_true"] = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)

    finite = pd.Series(train_scores).replace([np.inf, -np.inf], np.nan).dropna().values
    threshold = np.percentile(finite, threshold_percentile) if finite.size else np.inf

    segment_scores["y_pred"] = (segment_scores["score"] >= threshold).astype(int)

    y_true = segment_scores["y_true"]
    y_pred = segment_scores["y_pred"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, segment_scores["score"])

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"{sensor_type}")
    print(f"Threshold (Percentile {threshold_percentile}): {threshold:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    metrics = {"Sensors": sensor_type,
               "Accuracy": accuracy,
               "Precision": precision,
               "Recall": recall,
               "F1": f1,
               "ROC_AUC": roc_auc,
               "TN": cm[0, 0],
               "FP": cm[0, 1],
               "FN": cm[1, 0],
               "TP": cm[1, 1]}

    return metrics, segment_scores, model, scaler, threshold


In [10]:

def evaluate_acc_gyro_only(pred_acc, pred_gyro, title):
    """
    Funkcja łączy predykcje z akcelerometru i żyroskopu, ignorując mikrofon.
    Segment jest anomalią tylko, gdy oba czujniki (ACC i GYRO) wskażą anomalię (logiczne AND).
    """
    df_combined = (pred_acc[["segment_id", "y_true", "y_pred"]]
                   .rename(columns={"y_pred": "pred_acc"})
                   .merge(pred_gyro[["segment_id", "y_pred"]]
                          .rename(columns={"y_pred": "pred_gyro"}),
                          on="segment_id"))

    # Uznajemy za anomalię, jeśli oba modele wskażą 1
    df_combined["y_pred_final"] = ((df_combined["pred_acc"] == 1) & (df_combined["pred_gyro"] == 1)).astype(int)

    y_true = df_combined["y_true"]
    y_pred = df_combined["y_pred_final"]

    print(f"--- {title} ---")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-score : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print("-" * 40)

    return df_combined



Trenowanie
i
testowanie
osobnych
modeli(Autoenkodery)
wyłącznie
na
domenie
source

In [11]:

metrics_acc_source, pred_acc_source, model_acc_source, scaler_acc_source, threshold_acc_source = run_autoencoder_pipeline(
    X_train_source_acc,
    X_test_source_acc,
    X_train_df_source,
    X_test_df_source,
    "ACC - source",
)

metrics_mic_source, pred_mic_source, model_mic_source, scaler_mic_source, threshold_mic_source = run_autoencoder_pipeline(
    X_train_source_mic,
    X_test_source_mic,
    X_train_df_source,
    X_test_df_source,
    "MIC - source")

metrics_gyro_source, pred_gyro_source, model_gyro_source, scaler_gyro_source, threshold_gyro_source = run_autoencoder_pipeline(
    X_train_source_gyro,
    X_test_source_gyro,
    X_train_df_source,
    X_test_df_source,
    "GYRO - source")

ACC - source
Threshold (Percentile 99): 0.0038
Accuracy : 0.6638
Precision: 0.6532
Recall   : 0.6983
F1-score : 0.6750
ROC AUC  : 0.7403
[[73 43]
 [35 81]]
MIC - source
Threshold (Percentile 99): 0.0068
Accuracy : 0.5302
Precision: 0.5376
Recall   : 0.4310
F1-score : 0.4785
ROC AUC  : 0.5496
[[73 43]
 [66 50]]
GYRO - source
Threshold (Percentile 99): 0.0238
Accuracy : 0.5690
Precision: 0.5667
Recall   : 0.5862
F1-score : 0.5763
ROC AUC  : 0.5917
[[64 52]
 [48 68]]



Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię

In [12]:

final_source_results = (pred_acc_source[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                        .merge(pred_mic_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                               on="segment_id")
                        .merge(pred_gyro_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                               on="segment_id"))

final_source_results["y_pred_final"] = (
        final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)
y_true_source = final_source_results["y_true"]
y_pred_source_final = final_source_results["y_pred_final"]

accuracy_source = accuracy_score(y_true_source, y_pred_source_final)
precision_source = precision_score(y_true_source, y_pred_source_final, zero_division=0)
recall_source = recall_score(y_true_source, y_pred_source_final, zero_division=0)
f1_source = f1_score(y_true_source, y_pred_source_final, zero_division=0)
cm_source = confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1])

print("Final Source Results (at least 1 sensor):")
print(f"Accuracy : {accuracy_source:.4f}")
print(f"Precision: {precision_source:.4f}")
print(f"Recall   : {recall_source:.4f}")
print(f"F1-score : {f1_source:.4f}")
print("Confusion matrix:")
print(cm_source)

Final Source Results (at least 1 sensor):
Accuracy : 0.5431
Precision: 0.5253
Recall   : 0.8966
F1-score : 0.6624
Confusion matrix:
[[ 22  94]
 [ 12 104]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię

In [13]:

final_source_results["y_pred_final"] = (
        final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true_source = final_source_results["y_true"]
y_pred_source_final = final_source_results["y_pred_final"]

accuracy_source = accuracy_score(y_true_source, y_pred_source_final)
precision_source = precision_score(y_true_source, y_pred_source_final, zero_division=0)
recall_source = recall_score(y_true_source, y_pred_source_final, zero_division=0)
f1_source = f1_score(y_true_source, y_pred_source_final, zero_division=0)
cm_source = confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1])

print("Final Source Results (majority voting):")
print(f"Accuracy : {accuracy_source:.4f}")
print(f"Precision: {precision_source:.4f}")
print(f"Recall   : {recall_source:.4f}")
print(f"F1-score : {f1_source:.4f}")
print("Confusion matrix:")
print(cm_source)


Final Source Results (majority voting):
Accuracy : 0.6509
Precision: 0.6636
Recall   : 0.6121
F1-score : 0.6368
Confusion matrix:
[[80 36]
 [45 71]]


Ewaluacja
z
pominięciem
mikrofonu(Tylko
ACC
i
GYRO - logiczne
AND) - Source -> Source

In [14]:

final_acc_gyro_source = evaluate_acc_gyro_only(pred_acc_source, pred_gyro_source, "Source -> Source (ACC & GYRO Only)")


--- Source -> Source (ACC & GYRO Only) ---
Accuracy : 0.6509
Precision: 0.7333
Recall   : 0.4741
F1-score : 0.5759
Confusion matrix:
 [[96 20]
 [61 55]]
----------------------------------------


Trenowanie
osobnych
modeli
dla
każdego
czujnika
wyłącznie
z
wykorzystaniem
domeny
source, a
testowanie
na
target

In [15]:

metrics_acc, pred_acc, model_acc, scaler_acc, threshold_acc = run_autoencoder_pipeline(X_train_source_acc,
                                                                                       X_test_target_acc,
                                                                                       X_train_df_source,
                                                                                       X_test_df_target,
                                                                                       "ACC source-> target")
metrics_mic, pred_mic, model_mic, scaler_mic, threshold_mic = run_autoencoder_pipeline(X_train_source_mic,
                                                                                       X_test_target_mic,
                                                                                       X_train_df_source,
                                                                                       X_test_df_target,
                                                                                       "MIC source-> target")
metrics_gyro, pred_gyro, model_gyro, scaler_gyro, threshold_gyro = run_autoencoder_pipeline(X_train_source_gyro,
                                                                                            X_test_target_gyro,
                                                                                            X_train_df_source,
                                                                                            X_test_df_target,
                                                                                            "GYRO source-> target")

ACC source-> target
Threshold (Percentile 99): 0.0055
Accuracy : 0.5172
Precision: 0.5089
Recall   : 0.9828
F1-score : 0.6706
ROC AUC  : 0.5537
[[  6 110]
 [  2 114]]
MIC source-> target
Threshold (Percentile 99): 0.0058
Accuracy : 0.4871
Precision: 0.4348
Recall   : 0.0862
F1-score : 0.1439
ROC AUC  : 0.4775
[[103  13]
 [106  10]]
GYRO source-> target
Threshold (Percentile 99): 0.0133
Accuracy : 0.4828
Precision: 0.4886
Recall   : 0.7414
F1-score : 0.5890
ROC AUC  : 0.4131
[[26 90]
 [30 86]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię

In [16]:

final_results = (pred_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                 .merge(pred_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
                 .merge(pred_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))

final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)

y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results source -> target (at least 1 sensor):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results source -> target (at least 1 sensor):
Accuracy : 0.5043
Precision: 0.5022
Recall   : 0.9914
F1-score : 0.6667
Confusion matrix:
[[  2 114]
 [  1 115]]


Teraz
segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię

In [17]:


final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results source -> target (majority voting):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)


Final Results source -> target (majority voting):
Accuracy : 0.4828
Precision: 0.4888
Recall   : 0.7500
F1-score : 0.5918
Confusion matrix:
[[25 91]
 [29 87]]


Ewaluacja
z
pominięciem
mikrofonu(Tylko
ACC
i
GYRO - logiczne
AND) - Source -> Target

In [18]:

final_acc_gyro_tgt = evaluate_acc_gyro_only(pred_acc, pred_gyro, "Source -> Target (ACC & GYRO Only)")


--- Source -> Target (ACC & GYRO Only) ---
Accuracy : 0.4914
Precision: 0.4942
Recall   : 0.7328
F1-score : 0.5903
Confusion matrix:
 [[29 87]
 [31 85]]
----------------------------------------


Trenowanie
i
testowanie
osobnych
modeli
dla
każdego
czujnika
z
wykorzystaniem
całych
zbiorów
treningowego
i
testowego

In [19]:


metrics_acc, pred_acc, model_acc, scaler_acc, threshold_acc = run_autoencoder_pipeline(X_train_acc, X_test_acc,
                                                                                       X_train_df, X_test_df, "ACC")
metrics_mic, pred_mic, model_mic, scaler_mic, threshold_mic = run_autoencoder_pipeline(X_train_mic, X_test_mic,
                                                                                       X_train_df, X_test_df, "MIC")
metrics_gyro, pred_gyro, model_gyro, scaler_gyro, threshold_gyro = run_autoencoder_pipeline(X_train_gyro, X_test_gyro,
                                                                                            X_train_df, X_test_df,
                                                                                            "GYRO")


ACC
Threshold (Percentile 99): 0.0054
Accuracy : 0.6099
Precision: 0.5775
Recall   : 0.8190
F1-score : 0.6774
ROC AUC  : 0.6191
[[ 93 139]
 [ 42 190]]
MIC
Threshold (Percentile 99): 0.0107
Accuracy : 0.5323
Precision: 0.5591
Recall   : 0.3060
F1-score : 0.3955
ROC AUC  : 0.5415
[[176  56]
 [161  71]]
GYRO
Threshold (Percentile 99): 0.0284
Accuracy : 0.5129
Precision: 0.5111
Recall   : 0.5948
F1-score : 0.5498
ROC AUC  : 0.5244
[[100 132]
 [ 94 138]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię

In [20]:


final_results = (pred_acc[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                 .merge(pred_mic[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}), on="segment_id")
                 .merge(pred_gyro[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}), on="segment_id"))

final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)

y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results (at least 1 sensor):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)

Final Results (at least 1 sensor):
Accuracy : 0.5431
Precision: 0.5235
Recall   : 0.9612
F1-score : 0.6778
Confusion matrix:
[[ 29 203]
 [  9 223]]



Teraz
segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię


In [21]:

final_results["y_pred_final"] = (final_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_true = final_results["y_true"]
y_pred_final = final_results["y_pred_final"]

accuracy = accuracy_score(y_true, y_pred_final)
precision = precision_score(y_true, y_pred_final, zero_division=0)
recall = recall_score(y_true, y_pred_final, zero_division=0)
f1 = f1_score(y_true, y_pred_final, zero_division=0)
cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

print("Final Results (majority voting):")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print("Confusion matrix:")
print(cm)


Final Results (majority voting):
Accuracy : 0.5690
Precision: 0.5630
Recall   : 0.6164
F1-score : 0.5885
Confusion matrix:
[[121 111]
 [ 89 143]]


Ewaluacja
z
pominięciem
mikrofonu(Tylko
ACC
i
GYRO - logiczne
AND) - ALL -> ALL

In [22]:

final_acc_gyro_all = evaluate_acc_gyro_only(pred_acc, pred_gyro, "ALL -> ALL (ACC & GYRO Only)")

--- ALL -> ALL (ACC & GYRO Only) ---
Accuracy : 0.5582
Precision: 0.5692
Recall   : 0.4784
F1-score : 0.5199
Confusion matrix:
 [[148  84]
 [121 111]]
----------------------------------------
